In [18]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import os
import re
from datetime import datetime


## get the directories
base_dir = r'H:\My Drive\DroughtSurfing'
locations = ['WY', 'AZ', 'UT']
# locations = ['UT', 'AZ']

## define file paths for the exported GeoTIFFs within each directory
ndvi_file_names = [
    'smoothed_ndvi.tif'
]

In [16]:
def calculate_global_quantiles(ndvi_array, lower_percentile=2.5, upper_percentile=97.5, no_data_value=None):
    """
    Calculate the global lower and upper quantiles for the NDVI values, excluding no-data values.
    
    Args:
    - ndvi_array: The numpy array of NDVI values.
    - lower_percentile: The lower percentile to compute (default 2.5).
    - upper_percentile: The upper percentile to compute (default 97.5).
    - no_data_value: Value representing no-data (e.g., 0, -9999), if applicable.
    
    Returns:
    - lower_quantile: The value at the lower percentile.
    - upper_quantile: The value at the upper percentile.
    """
    ## if no_data_value is specified, mask those values
    if no_data_value is not None:
        ndvi_array = np.ma.masked_equal(ndvi_array, no_data_value)
        ndvi_valid = ndvi_array.compressed()  # Extract only valid (non-masked) values
    else:
        ndvi_valid = ndvi_array[~np.isnan(ndvi_array)]  # Remove NaN values (if present)

    ## ensure there are valid values to compute quantiles
    if len(ndvi_valid) == 0:
        return np.nan, np.nan
    
    ## calculate quantiles on valid values
    lower_quantile = np.percentile(ndvi_valid, lower_percentile)
    upper_quantile = np.percentile(ndvi_valid, upper_percentile)
    
    return lower_quantile, upper_quantile

def scale_ndvi(ndvi_array, lower_quantile, upper_quantile, no_data_value=None):
    """
    Scale NDVI values between 0 and 1 based on the global lower and upper quantiles.
    
    Args:
    - ndvi_array: The numpy array of NDVI values.
    - lower_quantile: The value at the lower percentile (e.g., 0.025 quantile).
    - upper_quantile: The value at the upper percentile (e.g., 0.975 quantile).
    - no_data_value: Value representing no-data (e.g., 0, -9999), if applicable.
    
    Returns:
    - scaled_ndvi: The NDVI values scaled between 0 and 1.
    """
    ## if no_data_value is specified, mask those values
    if no_data_value is not None:
        ndvi_array = np.ma.masked_equal(ndvi_array, no_data_value)
    
    ## scale NDVI using the formula: (NDVI - lower_quantile) / (upper_quantile - lower_quantile)
    scaled_ndvi = (ndvi_array - lower_quantile) / (upper_quantile - lower_quantile)
    
    ## ensure values are constrained between 0 and 1
    scaled_ndvi = np.clip(scaled_ndvi, 0, 1)
    
    return scaled_ndvi

def process_ndvi_files(base_dir, locations, ndvi_file_names):
    """
    Process the NDVI files by scaling them without saving the result.
    
    Args:
    - base_dir: The base directory containing NDVI files.
    - locations: List of location directories.
    - ndvi_file_names: List of NDVI file names to process.
    
    Returns:
    - scaled_ndvi_dict: A dictionary with scaled NDVI arrays keyed by location and file name.
    """
    scaled_ndvi_dict = {}
    
    for location in locations:
        location_dir = os.path.join(base_dir, location)
        
        for ndvi_file in ndvi_file_names:
            file_path = os.path.join(location_dir, ndvi_file)
            if not os.path.exists(file_path):
                print(f"File {file_path} does not exist.")
                continue
            
            ## open the NDVI GeoTIFF
            with rasterio.open(file_path) as src:
                ndvi_array = src.read(1)  # Read the first band (assuming it's NDVI)
                no_data_value = src.nodata  # Retrieve the no-data value from the file metadata
                
                ## calculate the global quantiles
                lower_quantile, upper_quantile = calculate_global_quantiles(ndvi_array, no_data_value=no_data_value)
                print(f"Lower Quantile (2.5%): {lower_quantile}")
                print(f"Upper Quantile (97.5%): {upper_quantile}")
                
                ## scale the NDVI values based on the quantiles
                scaled_ndvi = scale_ndvi(ndvi_array, lower_quantile, upper_quantile, no_data_value=no_data_value)
                
                ## store the result in the dictionary
                scaled_ndvi_dict[(location, ndvi_file)] = scaled_ndvi
    
    return scaled_ndvi_dict

## process the NDVI files and get the scaled arrays
scaled_ndvi_dict = process_ndvi_files(base_dir, locations, ndvi_file_names)


Lower Quantile (2.5%): 1430.0
Upper Quantile (97.5%): 4024.0


In [17]:

def extract_dates(band_names):
    """
    Extracts dates from a list of band names in the format 'YYYY_MM_DD'.
    
    Args:
    - band_names: A list of band names containing dates in 'YYYY_MM_DD' format.
    
    Returns:
    - dates: A list of datetime objects.
    """
    dates = []
    for name in band_names:

        ## extract the date part from the name using regex
        match = re.search(r'(\d{4}_\d{2}_\d{2})', name)
        if match:
            date_str = match.group(1)

            ## convert string to datetime object
            date_obj = datetime.strptime(date_str, '%Y_%m_%d')
            dates.append(date_obj)
    return dates

def save_scaled_ndvi_as_geotiff(scaled_ndvi, src, output_file):
    """
    Save the scaled NDVI array as a GeoTIFF, retaining the georeferencing information from the source image.
    
    Args:
    - scaled_ndvi: The numpy array of scaled NDVI values.
    - src: The source rasterio object to retain the georeferencing information.
    - output_file: The file path for the output GeoTIFF.
    """
    with rasterio.open(
        output_file, 
        'w', 
        driver='GTiff', 
        height=scaled_ndvi.shape[0], 
        width=scaled_ndvi.shape[1], 
        count=1, 
        dtype=scaled_ndvi.dtype,
        crs=src.crs, 
        transform=src.transform
    ) as dst:
        dst.write(scaled_ndvi, 1)

def process_and_save_scaled_ndvi(base_dir, locations, ndvi_file_names, band_names):
    """
    Process and scale the NDVI files, then save the scaled arrays as GeoTIFFs with the date in the file name.
    
    Args:
    - base_dir: The base directory containing NDVI files.
    - locations: List of location directories.
    - ndvi_file_names: List of NDVI file names to process.
    - band_names: The list of band names containing dates.
    """
    dates = extract_dates(band_names)
    
    for location in locations:
        location_dir = os.path.join(base_dir, location)
        
        for i, ndvi_file in enumerate(ndvi_file_names):
            file_path = os.path.join(location_dir, ndvi_file)
            if not os.path.exists(file_path):
                print(f"File {file_path} does not exist.")
                continue
            
            ## open the NDVI GeoTIFF
            with rasterio.open(file_path) as src:
                ndvi_array = src.read(1)  # Read the first band (assuming it's NDVI)
                no_data_value = src.nodata  # Retrieve the no-data value from the file metadata
                
                ## scale the NDVI values based on global quantiles or other logic
                lower_quantile = np.percentile(ndvi_array[ndvi_array != no_data_value], 2.5)
                upper_quantile = np.percentile(ndvi_array[ndvi_array != no_data_value], 97.5)
                scaled_ndvi = np.clip((ndvi_array - lower_quantile) / (upper_quantile - lower_quantile), 0, 1)
                
                ## create the output file name using the extracted date
                date_str = dates[i].strftime('%Y_%m_%d')
                output_file = os.path.join(location_dir, f"scaled_ndvi_{date_str}.tif")
                
                ## save the scaled NDVI as a new GeoTIFF
                save_scaled_ndvi_as_geotiff(scaled_ndvi, src, output_file)
                print(f"Saved scaled NDVI to {output_file}")

## process the files, scale them, and save as GeoTIFF
process_and_save_scaled_ndvi(base_dir, locations, ndvi_file_names, band_names)


File H:\My Drive\DroughtSurfing\UT\smoothed_ndvi_2022_01_01.tif does not exist.
File H:\My Drive\DroughtSurfing\UT\smoothed_ndvi_2022_01_15.tif does not exist.
File H:\My Drive\DroughtSurfing\UT\smoothed_ndvi_2022_02_01.tif does not exist.
